[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/064.%20The%20Cross-Docking%20Door%20Assignment%20Problem/P64-Tier-9_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# P64-Tier-9: QUBO-Based Cross-Docking Door Assignment

## Purpose and scope

This notebook formulates a cross-docking door assignment problem as a quadratic unconstrained binary optimization (QUBO) model and searches for a low-energy binary solution using a classical, quantum-inspired stochastic annealing procedure. The example has vehicles, doors, assignment costs, unit vehicle demands, and door capacities. The notebook constructs the model, runs the annealer, and checks whether the returned assignment satisfies the constraints.

The annealer in this notebook runs on a conventional computer. Its stochastic search is inspired by annealing ideas, but it does not simulate a quantum state, quantum tunneling, superposition, or entanglement, and it does not establish a quantum speedup or quantum advantage. The results are an algorithmic demonstration on one generated instance, not a performance guarantee.

### Model assumptions

- Each vehicle must be assigned to exactly one door.
- A vehicle's demand is an integer number of capacity units. The example uses demand one for every vehicle.
- Each door has a nonnegative integer capacity, and its total assigned demand may be less than or equal to that capacity.
- Assignment costs are additive: the cost of a complete assignment is the sum of the costs of its selected vehicle-door pairs.
- The model does not include travel times, vehicle-door compatibility, time windows, congestion, or interactions between assignments other than door capacity.

## 1. From assignment decisions to a QUBO

A binary decision variable records whether vehicle \(i\) uses door \(j\):

$$
x_{ij} =
\begin{cases}
1, & \text{if vehicle } i \text{ is assigned to door } j,\\
0, & \text{otherwise.}
\end{cases}
$$

Let \(c_{ij}\) be the assignment cost, \(d_i\) the demand of vehicle \(i\), and \(\mathrm{cap}_j\) the capacity of door \(j\). The assignment conditions are

$$
\sum_{j=1}^{M} x_{ij} = 1 \quad (i=1,\ldots,N),
\qquad
\sum_{i=1}^{N} d_i x_{ij} \leq \mathrm{cap}_j \quad (j=1,\ldots,M).
$$

The first condition requires one door per vehicle. The second limits the total demand assigned to each door. A QUBO has a binary objective and no explicit constraints, so the constraints are encoded as penalties in the objective.

The assignment condition is represented by a squared equality penalty. A capacity inequality is converted to an equality by introducing nonnegative integer slack \(s_j\), measured in capacity units:

$$
\sum_{i=1}^{N} d_i x_{ij} + s_j = \mathrm{cap}_j,
\qquad s_j \in \{0,1,\ldots,\mathrm{cap}_j\}.
$$

The slack is the unused capacity. It is encoded using binary variables \(y_{jk}\) with weights \(w_{jk}\), so \(s_j=\sum_k w_{jk}y_{jk}\). The model minimized by the QUBO constructor is

$$
\begin{aligned}
E(x,y) ={}& \sum_{i=1}^{N}\sum_{j=1}^{M} c_{ij}x_{ij}\\
&+ \lambda_{\mathrm{assignment}}\sum_{i=1}^{N}
  \left(\sum_{j=1}^{M}x_{ij}-1\right)^2\\
&+ \lambda_{\mathrm{capacity}}\sum_{j=1}^{M}
  \left(\sum_{i=1}^{N}d_i x_{ij}+\sum_k w_{jk}y_{jk}-\mathrm{cap}_j\right)^2.
\end{aligned}
$$

Here \(N\) is the number of vehicles, \(M\) is the number of doors, \(c_{ij}\) is the cost matrix entry, and the two \(\lambda\) values are nonnegative penalty weights. When all constraints are satisfied, both penalty terms are zero and the energy equals the assignment cost. A finite penalty weight does not automatically guarantee that every minimum-energy binary vector is feasible; feasibility must therefore be checked after the search.

For a binary vector \(z\), the constructed model is stored in the convention

$$
E(z)=z^{\mathsf T}Qz+h^{\mathsf T}z+c_0,
$$

where \(Q\) contains quadratic coefficients, \(h\) contains linear coefficients, and \(c_0\) is a constant. The implementation uses a symmetric \(Q\); consequently, each off-diagonal pair contributes through both \(Q_{ab}z_az_b\) and \(Q_{ba}z_bz_a\).

## 2. Stochastic annealing search

The search maintains a current binary vector and proposes nearby vectors by flipping bits. A candidate with lower energy is accepted. A higher-energy candidate may be accepted probabilistically, with a probability that decreases as the temperature cools and as the energy increase grows. This allows the search to explore beyond immediate improvements; it does not guarantee a global optimum.

The procedure starts from a random vector unless an initial vector is supplied, evaluates the QUBO energy, and repeats candidate generation and acceptance for a fixed number of steps. The best vector seen is returned. The temperature schedule and stochastic update rules are classical heuristics, not a numerical simulation of quantum dynamics.

With \(n\) binary variables and \(T\) search steps, the dense matrix-vector energy evaluation used here takes \(O(n^2)\) time per candidate, giving \(O(Tn^2)\) time for the main search, apart from occasional multi-bit exploration and coupling analysis. The number of binary states is \(2^n\), so the search space grows exponentially with \(n\).

## 3. Reading the results

The reported QUBO energy includes assignment costs and constraint penalties; it is not, by itself, the operational assignment cost. The notebook reports assignment cost separately and verifies one-door-per-vehicle and door-capacity conditions. A feasible result can be interpreted as a valid assignment for the stated model. A single stochastic run is not proof of optimality, and its timing and solution quality should not be interpreted as evidence of quantum advantage.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Fixed seeds make the generated instance and stochastic search reproducible.
np.random.seed(64)
random.seed(64)

print("QUBO assignment example is ready.")

In [ ]:
@dataclass
class CrossDockingInstance:
    """Input data for the cross-docking door assignment problem."""

    num_vehicles: int
    num_doors: int
    cost_matrix: np.ndarray
    door_capacities: List[int]
    vehicle_demands: List[int]


@dataclass
class QUBOFormulation:
    """QUBO coefficients and mappings back to assignment decisions."""

    Q_matrix: np.ndarray
    linear_terms: np.ndarray
    constant_term: float
    num_variables: int
    variable_mapping: Dict[Tuple[int, int], int]
    slack_mapping: Dict[int, List[Tuple[int, int]]]
    num_vehicles: int
    num_doors: int
    door_capacities: List[int]
    vehicle_demands: List[int]


@dataclass
class QuantumAnnealingResult:
    """Best binary vector found by the stochastic annealing search."""

    solution_vector: np.ndarray
    objective_value: float
    assignment_matrix: np.ndarray
    is_feasible: bool
    computation_time: float
    quantum_metrics: Dict[str, float]
    energy_history: List[float]


print("Problem and result data structures are defined.")

In [ ]:
class QUBOConstructor:
    """Build and evaluate a QUBO model for the assignment instance."""

    def __init__(
        self,
        penalty_weight_assignment: float = 100.0,
        penalty_weight_capacity: float = 100.0,
    ) -> None:
        if penalty_weight_assignment < 0 or penalty_weight_capacity < 0:
            raise ValueError("Penalty weights must be nonnegative.")
        self.lambda_assignment = penalty_weight_assignment
        self.lambda_capacity = penalty_weight_capacity

    def construct_qubo(self, instance: CrossDockingInstance) -> QUBOFormulation:
        """Encode assignment equalities and capacity inequalities as penalties."""
        n_vehicles = instance.num_vehicles
        n_doors = instance.num_doors

        if instance.cost_matrix.shape != (n_vehicles, n_doors):
            raise ValueError("The cost matrix dimensions do not match the instance.")
        if len(instance.door_capacities) != n_doors:
            raise ValueError("One capacity value is required for each door.")
        if len(instance.vehicle_demands) != n_vehicles:
            raise ValueError("One demand value is required for each vehicle.")
        if any(capacity < 0 for capacity in instance.door_capacities):
            raise ValueError("Door capacities must be nonnegative.")
        if any(demand < 0 for demand in instance.vehicle_demands):
            raise ValueError("Vehicle demands must be nonnegative.")
        if any(int(value) != value for value in instance.door_capacities):
            raise ValueError("Door capacities must be integers for binary slack encoding.")
        if any(int(value) != value for value in instance.vehicle_demands):
            raise ValueError("Vehicle demands must be integers for this model.")

        variable_mapping = {
            (i, j): i * n_doors + j
            for i in range(n_vehicles)
            for j in range(n_doors)
        }
        next_index = n_vehicles * n_doors
        slack_mapping: Dict[int, List[Tuple[int, int]]] = {}

        # Binary weights represent every integer unused-capacity value from zero to capacity.
        for j, capacity_value in enumerate(instance.door_capacities):
            capacity = int(capacity_value)
            weights: List[int] = []
            remaining = capacity
            weight = 1
            while remaining > 0:
                current_weight = min(weight, remaining)
                weights.append(current_weight)
                remaining -= current_weight
                weight *= 2

            slack_mapping[j] = []
            for bit_position, bit_weight in enumerate(weights):
                slack_mapping[j].append((next_index, bit_weight))
                next_index += 1

        n_variables = next_index
        Q_matrix = np.zeros((n_variables, n_variables), dtype=float)
        linear_terms = np.zeros(n_variables, dtype=float)
        constant_term = 0.0

        # Assignment costs are linear because each decision variable is binary.
        for (i, j), var_idx in variable_mapping.items():
            linear_terms[var_idx] += instance.cost_matrix[i, j]

        # Expand lambda * (sum(a_k z_k) - target)^2 into QUBO coefficients.
        def add_squared_equality(
            terms: List[Tuple[int, float]], target: float, penalty: float
        ) -> None:
            nonlocal constant_term
            constant_term += penalty * target**2

            for var_idx, coefficient in terms:
                # Since z^2 = z for binary z, its squared term is linear.
                linear_terms[var_idx] += penalty * (
                    coefficient**2 - 2 * target * coefficient
                )

            # Each unordered pair is stored symmetrically; z.T @ Q @ z counts it twice.
            for position, (idx_a, coefficient_a) in enumerate(terms):
                for idx_b, coefficient_b in terms[position + 1 :]:
                    interaction = penalty * coefficient_a * coefficient_b
                    Q_matrix[idx_a, idx_b] += interaction
                    Q_matrix[idx_b, idx_a] += interaction

        for i in range(n_vehicles):
            assignment_terms = [
                (variable_mapping[(i, j)], 1.0) for j in range(n_doors)
            ]
            add_squared_equality(assignment_terms, 1.0, self.lambda_assignment)

        for j in range(n_doors):
            capacity_terms = [
                (
                    variable_mapping[(i, j)],
                    float(instance.vehicle_demands[i]),
                )
                for i in range(n_vehicles)
            ]
            capacity_terms.extend(
                (var_idx, float(bit_weight))
                for var_idx, bit_weight in slack_mapping[j]
            )
            add_squared_equality(
                capacity_terms,
                float(instance.door_capacities[j]),
                self.lambda_capacity,
            )

        return QUBOFormulation(
            Q_matrix=Q_matrix,
            linear_terms=linear_terms,
            constant_term=constant_term,
            num_variables=n_variables,
            variable_mapping=variable_mapping,
            slack_mapping=slack_mapping,
            num_vehicles=n_vehicles,
            num_doors=n_doors,
            door_capacities=list(instance.door_capacities),
            vehicle_demands=list(instance.vehicle_demands),
        )

    def evaluate_qubo(
        self, qubo: QUBOFormulation, solution_vector: np.ndarray
    ) -> float:
        """Return the full QUBO energy, including its constant term."""
        if solution_vector.shape != (qubo.num_variables,):
            raise ValueError("The solution vector has the wrong number of variables.")
        quadratic = solution_vector @ qubo.Q_matrix @ solution_vector
        linear = qubo.linear_terms @ solution_vector
        return float(quadratic + linear + qubo.constant_term)


print("QUBO constructor is defined.")

In [ ]:
class QuantumAnnealer:
    """Classical stochastic annealer operating on a binary QUBO vector."""

    def __init__(
        self,
        temperature_schedule: str = "linear",
        tunneling_probability: float = 0.1,
    ) -> None:
        if temperature_schedule not in {"linear", "exponential", "polynomial"}:
            raise ValueError("Unsupported temperature schedule.")
        if not 0.0 <= tunneling_probability <= 1.0:
            raise ValueError("The uphill-move probability must be between zero and one.")
        self.temperature_schedule = temperature_schedule
        self.tunneling_prob = tunneling_probability

    def anneal(
        self,
        qubo: QUBOFormulation,
        num_steps: int = 1000,
        initial_solution: Optional[np.ndarray] = None,
    ) -> QuantumAnnealingResult:
        """Search for a low-energy vector using stochastic bit-flip proposals."""
        if num_steps <= 0:
            raise ValueError("num_steps must be positive.")
        start_time = time.time()

        if initial_solution is None:
            current_solution = np.random.randint(0, 2, qubo.num_variables)
        else:
            current_solution = np.asarray(initial_solution, dtype=int).copy()
            if current_solution.shape != (qubo.num_variables,):
                raise ValueError("The initial solution has the wrong number of variables.")
            if not np.all(np.isin(current_solution, [0, 1])):
                raise ValueError("The initial solution must be binary.")

        current_energy = self._evaluate_energy(qubo, current_solution)
        best_solution = current_solution.copy()
        best_energy = current_energy
        energy_history = [current_energy]
        uphill_moves_accepted = 0
        multi_bit_improvements = 0
        coupled_move_improvements = 0

        for step in range(num_steps):
            temperature = self._calculate_temperature(step, num_steps)
            candidate_solution = self._generate_candidate(
                current_solution, temperature, qubo.num_variables
            )
            candidate_energy = self._evaluate_energy(qubo, candidate_solution)
            delta_energy = candidate_energy - current_energy

            if delta_energy < 0:
                current_solution = candidate_solution
                current_energy = candidate_energy
            elif np.random.random() < self.tunneling_prob:
                # This classical probabilistic rule sometimes permits uphill moves.
                acceptance_probability = np.exp(
                    -delta_energy / max(temperature, 1e-10)
                )
                if np.random.random() < acceptance_probability:
                    current_solution = candidate_solution
                    current_energy = candidate_energy
                    uphill_moves_accepted += 1

            if np.random.random() < 0.05:
                multi_bit_solution = self._superposition_exploration(
                    current_solution, qubo, temperature
                )
                multi_bit_energy = self._evaluate_energy(qubo, multi_bit_solution)
                if multi_bit_energy < current_energy:
                    current_solution = multi_bit_solution
                    current_energy = multi_bit_energy
                    multi_bit_improvements += 1

            if np.random.random() < 0.03:
                coupled_solution = self._entanglement_update(current_solution, qubo)
                coupled_energy = self._evaluate_energy(qubo, coupled_solution)
                if coupled_energy < current_energy:
                    current_solution = coupled_solution
                    current_energy = coupled_energy
                    coupled_move_improvements += 1

            if current_energy < best_energy:
                best_solution = current_solution.copy()
                best_energy = current_energy

            energy_history.append(current_energy)

        assignment_matrix = self._solution_to_assignment(qubo, best_solution)
        is_feasible = self._check_feasibility(assignment_matrix, qubo)
        initial_energy = energy_history[0]
        quantum_metrics = {
            "uphill_moves_accepted": uphill_moves_accepted,
            "multi_bit_improvements": multi_bit_improvements,
            "coupled_move_improvements": coupled_move_improvements,
            "energy_reduction": float(initial_energy - best_energy),
            "convergence_rate": self._calculate_convergence_rate(energy_history),
        }

        return QuantumAnnealingResult(
            solution_vector=best_solution,
            objective_value=float(best_energy),
            assignment_matrix=assignment_matrix,
            is_feasible=is_feasible,
            computation_time=time.time() - start_time,
            quantum_metrics=quantum_metrics,
            energy_history=energy_history,
        )

    def _calculate_temperature(self, step: int, total_steps: int) -> float:
        """Return the selected cooling schedule at the current step."""
        progress = step / total_steps
        if self.temperature_schedule == "linear":
            return 1.0 - progress
        if self.temperature_schedule == "exponential":
            return float(np.exp(-5.0 * progress))
        return (1.0 - progress) ** 2

    def _generate_candidate(
        self, current_solution: np.ndarray, temperature: float, num_variables: int
    ) -> np.ndarray:
        """Flip one or more bits, with the number of flips tied to temperature."""
        candidate = current_solution.copy()
        num_flips = min(num_variables, max(1, int(temperature * 5)))
        flip_indices = np.random.choice(num_variables, num_flips, replace=False)
        candidate[flip_indices] = 1 - candidate[flip_indices]
        return candidate

    def _evaluate_energy(
        self, qubo: QUBOFormulation, solution: np.ndarray
    ) -> float:
        """Evaluate the QUBO energy used to accept or reject a candidate."""
        quadratic = solution @ qubo.Q_matrix @ solution
        linear = qubo.linear_terms @ solution
        return float(quadratic + linear + qubo.constant_term)

    def _superposition_exploration(
        self,
        current_solution: np.ndarray,
        qubo: QUBOFormulation,
        temperature: float,
    ) -> np.ndarray:
        """Try a random multi-bit move as a classical neighborhood exploration."""
        candidate = current_solution.copy()
        max_flips = min(4, qubo.num_variables)
        min_flips = min(2, max_flips)
        num_flips = np.random.randint(min_flips, max_flips + 1)
        flip_indices = np.random.choice(qubo.num_variables, num_flips, replace=False)
        candidate[flip_indices] = 1 - candidate[flip_indices]
        return candidate

    def _entanglement_update(
        self, current_solution: np.ndarray, qubo: QUBOFormulation
    ) -> np.ndarray:
        """Try a joint flip of a pair with a nonzero QUBO interaction."""
        candidate = current_solution.copy()
        upper_triangle = np.triu(qubo.Q_matrix, k=1)
        coupled_pairs = np.argwhere(upper_triangle != 0)
        if len(coupled_pairs) > 0:
            first, second = coupled_pairs[np.random.randint(len(coupled_pairs))]
            candidate[first] = 1 - candidate[first]
            candidate[second] = 1 - candidate[second]
        return candidate

    def _solution_to_assignment(
        self, qubo: QUBOFormulation, solution: np.ndarray
    ) -> np.ndarray:
        """Extract vehicle-door decisions, excluding capacity slack bits."""
        assignment = np.zeros((qubo.num_vehicles, qubo.num_doors), dtype=int)
        for (vehicle, door), var_idx in qubo.variable_mapping.items():
            assignment[vehicle, door] = solution[var_idx]
        return assignment

    def _check_feasibility(
        self, assignment_matrix: np.ndarray, qubo: QUBOFormulation
    ) -> bool:
        """Check one-door-per-vehicle and demand-weighted door capacities."""
        if not np.all(np.sum(assignment_matrix, axis=1) == 1):
            return False
        door_loads = np.asarray(qubo.vehicle_demands) @ assignment_matrix
        return bool(np.all(door_loads <= np.asarray(qubo.door_capacities)))

    def _calculate_convergence_rate(self, energy_history: List[float]) -> float:
        """Estimate late-run energy decrease per iteration from a linear fit."""
        final_count = max(2, len(energy_history) // 10)
        final_energies = np.asarray(energy_history[-final_count:], dtype=float)
        if len(final_energies) < 2:
            return 0.0
        iteration_indices = np.arange(len(final_energies))
        slope = np.polyfit(iteration_indices, final_energies, 1)[0]
        return float(max(0.0, -slope))


print("Stochastic QUBO annealer is defined.")

## 4. Create the example instance

The cost matrix contains the cost of assigning each vehicle to each door. The random seed is fixed in the setup cell, so the generated matrix is reproducible. The total available capacity is a feasibility check on the instance as a whole; individual door limits are enforced in the QUBO and checked against the returned assignment.

In [ ]:
num_vehicles = 8
num_doors = 6

cost_matrix = np.random.uniform(10, 100, (num_vehicles, num_doors))
door_capacities = [2, 3, 2, 3, 2, 2]
vehicle_demands = [1] * num_vehicles

instance = CrossDockingInstance(
    num_vehicles=num_vehicles,
    num_doors=num_doors,
    cost_matrix=cost_matrix,
    door_capacities=door_capacities,
    vehicle_demands=vehicle_demands,
)

assert cost_matrix.shape == (num_vehicles, num_doors)
assert sum(vehicle_demands) <= sum(door_capacities)

print("Cross-docking instance created.")
print(f"Vehicles: {num_vehicles}")
print(f"Doors: {num_doors}")
print(f"Total demand: {sum(vehicle_demands)} capacity units")
print(f"Total door capacity: {sum(door_capacities)} capacity units")

cost_df = pd.DataFrame(
    cost_matrix,
    columns=[f"Door {j + 1}" for j in range(num_doors)],
    index=[f"Vehicle {i + 1}" for i in range(num_vehicles)],
)
display(cost_df.round(2))

plt.figure(figsize=(10, 6))
sns.heatmap(
    cost_matrix,
    annot=True,
    fmt=".1f",
    cmap="YlOrRd",
    xticklabels=[f"Door {j + 1}" for j in range(num_doors)],
    yticklabels=[f"Vehicle {i + 1}" for i in range(num_vehicles)],
    cbar_kws={"label": "Assignment cost"},
)
plt.title("Vehicle-to-door assignment costs")
plt.xlabel("Door")
plt.ylabel("Vehicle")
plt.tight_layout()
plt.show()

## 5. Construct and inspect the QUBO

The assignment variables are followed by binary slack variables. Slack allows each capacity constraint to represent any unused-capacity value from zero to the door's capacity. The QUBO energy can then be evaluated directly for any binary vector, including vectors that violate the assignment constraints.

In [ ]:
qubo_constructor = QUBOConstructor(
    penalty_weight_assignment=100.0,
    penalty_weight_capacity=50.0,
)
qubo = qubo_constructor.construct_qubo(instance)

print("QUBO formulation constructed.")
print(f"Binary variables, including slack: {qubo.num_variables}")
print(f"Assignment variables: {len(qubo.variable_mapping)}")
print(f"Q-matrix shape: {qubo.Q_matrix.shape}")
print(f"Constant term: {qubo.constant_term:.2f}")
print(f"Q-matrix is symmetric: {np.allclose(qubo.Q_matrix, qubo.Q_matrix.T)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
matrix_image = axes[0].imshow(qubo.Q_matrix, cmap="coolwarm", aspect="auto")
axes[0].set_title("QUBO quadratic coefficient matrix")
axes[0].set_xlabel("Binary variable index")
axes[0].set_ylabel("Binary variable index")
fig.colorbar(matrix_image, ax=axes[0], label="Quadratic coefficient")

axes[1].bar(np.arange(qubo.num_variables), qubo.linear_terms, alpha=0.8)
axes[1].set_title("QUBO linear coefficients")
axes[1].set_xlabel("Binary variable index")
axes[1].set_ylabel("Linear coefficient")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

random_solution = np.random.randint(0, 2, qubo.num_variables)
random_energy = qubo_constructor.evaluate_qubo(qubo, random_solution)
print(f"Energy of a random binary vector: {random_energy:.2f}")
print(f"Number of active binary variables: {int(np.sum(random_solution))}")

## 6. Run the stochastic annealing search

The returned objective value is the QUBO energy of the best vector encountered. The assignment matrix contains only vehicle-door variables; slack bits are used to encode unused capacity and are not displayed as assignments. The explicit feasibility check is important because a low-energy vector is not guaranteed to satisfy every constraint when penalty weights are finite.

In [ ]:
quantum_annealer = QuantumAnnealer(
    temperature_schedule="exponential",
    tunneling_probability=0.15,
)

result = quantum_annealer.anneal(
    qubo=qubo,
    num_steps=2000,
    initial_solution=None,
)

assignment_cost = float(np.sum(result.assignment_matrix * cost_matrix))
vehicle_assignments = np.sum(result.assignment_matrix, axis=1)
door_loads = np.asarray(vehicle_demands) @ result.assignment_matrix

assert np.isclose(
    result.objective_value,
    qubo_constructor.evaluate_qubo(qubo, result.solution_vector),
)
assert result.is_feasible == bool(
    np.all(vehicle_assignments == 1)
    and np.all(door_loads <= np.asarray(door_capacities))
)

print("Stochastic QUBO annealing run completed.")
print(f"QUBO energy: {result.objective_value:.2f}")
print(f"Assignment cost: {assignment_cost:.2f}")
print(f"Computation time: {result.computation_time:.3f} seconds")
print(f"Assignment is feasible: {result.is_feasible}")
print(f"Active binary variables, including slack: {int(np.sum(result.solution_vector))}")

print("\nSearch metrics:")
for metric_name, metric_value in result.quantum_metrics.items():
    print(f"  {metric_name.replace('_', ' ').title()}: {metric_value:.4f}")

assignment_df = pd.DataFrame(
    result.assignment_matrix,
    columns=[f"Door {j + 1}" for j in range(num_doors)],
    index=[f"Vehicle {i + 1}" for i in range(num_vehicles)],
)
print("\nVehicle-door assignment matrix:")
display(assignment_df)
print("Door loads:")
for j, load in enumerate(door_loads):
    print(f"  Door {j + 1}: {int(load)} / {door_capacities[j]} capacity units")

print("\nSelected assignments:")
for i, j in zip(*np.where(result.assignment_matrix == 1)):
    print(f"  Vehicle {i + 1} -> Door {j + 1} (cost: {cost_matrix[i, j]:.2f})")

## 7. Inspect the search trajectory and assignment

The energy trace records the current state's energy after each iteration, whereas the reported result is the best state encountered. The trace helps show the behavior of this stochastic run; it does not establish convergence to a global optimum. The assignment plot displays selected doors and the cost associated with each selected vehicle-door pair.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(result.energy_history, color="navy", linewidth=1)
axes[0].axhline(
    result.objective_value,
    color="darkorange",
    linestyle="--",
    label="Best energy encountered",
)
axes[0].set_title("Current-state QUBO energy during the search")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("QUBO energy")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

assignment_image = axes[1].imshow(result.assignment_matrix, cmap="Blues", aspect="auto")
axes[1].set_title("Selected vehicle-to-door assignments")
axes[1].set_xlabel("Door")
axes[1].set_ylabel("Vehicle")
axes[1].set_xticks(range(num_doors), [f"Door {j + 1}" for j in range(num_doors)])
axes[1].set_yticks(range(num_vehicles), [f"Vehicle {i + 1}" for i in range(num_vehicles)])
fig.colorbar(assignment_image, ax=axes[1], ticks=[0, 1], label="Assignment (0 or 1)")
plt.tight_layout()
plt.show()

print(f"Best QUBO energy encountered: {result.objective_value:.2f}")
print(f"Assignment cost, excluding penalties: {assignment_cost:.2f}")
print(f"Feasibility status: {'feasible' if result.is_feasible else 'infeasible'}")
print("This single run is a heuristic result, not an optimality certificate.")

## 8. Interpretation and limitations

The QUBO combines the operational assignment cost with penalties for violating one-door-per-vehicle and door-capacity conditions. For a feasible vector, these penalties vanish, so the QUBO energy equals the assignment cost. For an infeasible vector, the energy also reflects penalty terms, which is why the two values are reported separately.

The stochastic search uses bit flips, probabilistic acceptance of some uphill moves, occasional multi-bit proposals, and occasional joint flips of coupled variables. These are classical search mechanisms. Names such as `QuantumAnnealer`, `superposition_exploration`, and `entanglement_update` are retained as implementation labels, but they do not represent physical quantum processes.

A single run on one seeded instance cannot establish that the method is globally optimal, faster than another method, scalable to larger instances, or quantum-advantaged. The feasibility check establishes only whether the returned assignment satisfies the constraints encoded by this instance. The penalty weights and finite search budget also influence which solution is found.

## 9. Conclusion

This notebook develops a QUBO representation of the cross-docking door assignment problem and applies a classical stochastic annealing heuristic to it. Binary assignment variables represent vehicle-door choices, binary slack variables convert door-capacity inequalities into equalities, and squared penalties incorporate the constraints into the energy function.

The output should be interpreted in three parts: the QUBO energy measures the penalized objective, the assignment cost measures the selected operational costs, and the feasibility flag checks the stated assignment and capacity requirements. The search can find a useful candidate, but it does not provide an optimality certificate. No quantum hardware, quantum-state simulation, or quantum speedup is demonstrated.

The model is intentionally limited to additive assignment costs and door capacities. Additional operational requirements can be represented by extending the model, provided that their variables and constraints are encoded consistently in the QUBO.